# Notebook 7 — Explainability Pipeline Demonstration

**Goal:** Demonstrate the full interpretability pipeline:
1. Train IF + LSTM on normal data
2. Inject 3 specific anomalies (temp spike, fan drop, hashrate crash)
3. Show SHAP summaries from Isolation Forest
4. Show LSTM per-feature error heatmaps
5. Show fused explanations with confidence and failure signatures
6. Compare old z-score explanations vs new SHAP+LSTM explanations

**This notebook is fully runnable** — it generates its own data and trains models in-memory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Generate synthetic data
from notebooks import __path__ as _nb  # this might fail — alternative below
try:
    from notebooks.a]0_synthetic_data_generator import generate_normal_hour
except ImportError:
    exec(open('00_synthetic_data_generator.py').read())

from datetime import datetime

# Generate 4 hours of normal data for training
normal_data = generate_normal_hour(datetime(2024,1,1), samples=480, ambient_c=22.0)
print(f'Training data: {len(normal_data)} samples, {len(normal_data.columns)} columns')

## 1. Train both models on normal data

In [ ]:
from backend.ml.isolation_forest import IsolationForestDetector
from backend.ml.lstm_autoencoder import LSTMDetector
from backend.ml.explainer import AnomalyExplainer
from backend.ml.baseline import compute_baseline

# Convert DataFrame rows to list of dicts (same format as production)
feature_cols = [c for c in normal_data.columns
                if c not in ('timestamp','anomaly_label','anomaly_type')]
readings = normal_data[feature_cols].to_dict('records')

# Compute baseline
baseline = compute_baseline(readings)
print(f'Baseline computed for {len(baseline)} features')

# Train Isolation Forest
if_model = IsolationForestDetector('demo')
if_result = if_model.train(readings)
print(f'IF trained: {if_result["n_features"]} features, '
      f'yellow@{if_result["threshold_yellow"]:.3f}, '
      f'red@{if_result["threshold_red"]:.3f}, '
      f'SHAP={if_result.get("shap_available", False)}')

# Train LSTM
lstm_model = LSTMDetector('demo')
lstm_result = lstm_model.train(readings)
if 'error' in lstm_result:
    print(f'LSTM: {lstm_result["error"]}')
else:
    print(f'LSTM trained: {lstm_result["n_windows"]} windows, threshold={lstm_result["threshold"]:.6f}')

explainer = AnomalyExplainer()

## 2. Create 3 specific test anomalies

Each represents a distinct real-world failure mode.

In [ ]:
# Anomaly 1: Temperature spike — ambient temp rises, all chips overheat
temp_spike = dict(readings[-1])  # start from a normal reading
temp_spike.update({
    'temp1': 72.0, 'temp2': 71.0, 'temp3': 73.0, 'temp4': 70.0,
    'temp2_1': 82.0, 'temp2_2': 80.0, 'temp2_3': 83.0, 'temp2_4': 81.0,
    'temp_max': 83.0,
    'fan1': 3200, 'fan2': 3250,  # fans speed up to compensate
})

# Anomaly 2: Fan drop — fan 1 is dying
fan_drop = dict(readings[-1])
fan_drop.update({
    'fan1': 800,     # critical: below 1200 RPM
    'fan2': 2400,    # fan 2 still fine
    'temp2_1': 58.0, # temps starting to rise
    'temp2_2': 56.0,
    'temp_max': 58.0,
})

# Anomaly 3: Hashrate crash — board 3 lost chips
hash_crash = dict(readings[-1])
hash_crash.update({
    'chain_acn3': 45,           # lost 27 chips
    'chain_rate3': 20.0,         # hashrate crashed on board 3
    'chain_hw3': 35,             # HW errors spiking
    'GHS 5s': 210.0,             # total hashrate down
})

test_cases = [
    ('Temperature Spike (cooling failure)', temp_spike),
    ('Fan Drop (fan 1 dying)',              fan_drop),
    ('Hashrate Crash (board 3 chip loss)',  hash_crash),
]
print(f'Created {len(test_cases)} test anomalies')

## 3. Score each anomaly and show SHAP attributions

In [ ]:
for name, anomaly_reading in test_cases:
    print(f'\n{"="*60}')
    print(f'ANOMALY: {name}')
    print(f'{"="*60}')

    # ── IF scoring ────────────────────────────────────────────────────
    if_r = if_model.score(anomaly_reading)
    print(f'\nIF Score: {if_r["anomaly_score"]:.3f} ({if_r["severity"]})')

    if if_r['shap_features']:
        print('\nSHAP Feature Attributions (top 8):')
        print(f'{"Feature":25} {"SHAP":>8} {"Dir":>8} {"Value":>10} {"TrainMean":>10} {"Deviation":>10}')
        for f in if_r['shap_features'][:8]:
            print(f'{f["feature"]:25} {f["shap_value"]:+8.4f} {f["direction"]:>8} '
                  f'{f["value"]:10.2f} {f["train_mean"]:10.2f} {f["deviation"]:+10.2f}')

        # SHAP bar chart
        top = if_r['shap_features'][:8]
        fig, ax = plt.subplots(figsize=(10, 3.5))
        colors = ['#ef4444' if f['direction']=='anomaly' else '#3b82f6' for f in top]
        ax.barh([f['feature'] for f in top], [f['shap_value'] for f in top], color=colors)
        ax.invert_yaxis(); ax.axvline(0, color='black', linewidth=0.5)
        ax.set_xlabel('SHAP value (red = pushes toward anomaly)')
        ax.set_title(f'SHAP: {name}')
        plt.tight_layout(); plt.show()
    else:
        print('  (SHAP not available — install: pip install shap)')

## 4. LSTM per-feature error heatmap

In [ ]:
if lstm_model.is_trained:
    # Build a recent window for each anomaly
    # Use 19 normal readings + 1 anomaly at the end
    for name, anomaly_reading in test_cases:
        window = readings[-19:] + [anomaly_reading]  # 20 readings
        lstm_r = lstm_model.score_window(window)
        print(f'\n{name}:')
        print(f'  LSTM aggregate error: {lstm_r["lstm_error"]:.6f} '
              f'(threshold: {lstm_r["threshold"]:.6f}) '
              f'→ {"ANOMALY" if lstm_r["is_anomaly"] else "normal"}')
        if lstm_r['per_feature_error']:
            print(f'  Per-feature error ranking:')
            for f in lstm_r['per_feature_error']:
                print(f'    #{f["rank"]:2d} {f["feature"]:25} '
                      f'error={f["error"]:.6f} normalized={f["error_normalized"]:+.3f}')

    # Heatmap of per-feature errors across all 3 anomalies
    all_errors = {}
    for name, anomaly_reading in test_cases:
        window = readings[-19:] + [anomaly_reading]
        lstm_r = lstm_model.score_window(window)
        for f in lstm_r.get('per_feature_error', []):
            all_errors.setdefault(f['feature'], {})[name.split('(')[0].strip()] = f['error_normalized']

    if all_errors:
        heatmap_df = pd.DataFrame(all_errors).T
        fig, ax = plt.subplots(figsize=(8, max(4, len(heatmap_df) * 0.3)))
        sns.heatmap(heatmap_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
                    cbar_kws={'label': 'Normalized reconstruction error'})
        ax.set_title('LSTM per-feature reconstruction error across anomaly types')
        plt.tight_layout(); plt.show()
else:
    print('LSTM not trained (PyTorch may not be installed)')

## 5. Fused explanations

The fusion layer combines SHAP + LSTM errors, computes confidence based on model agreement, and maps to failure signatures.

In [ ]:
for name, anomaly_reading in test_cases:
    if_r = if_model.score(anomaly_reading)
    window = readings[-19:] + [anomaly_reading]
    lstm_r = lstm_model.score_window(window) if lstm_model.is_trained else {
        'lstm_error': 0, 'is_anomaly': False, 'per_feature_error': []}

    expl = explainer.explain(if_r, lstm_r, baseline, top_k=5)

    print(f'\n{"="*70}')
    print(f'FUSED EXPLANATION: {name}')
    print(f'{"="*70}')
    print(f'\nNarrative: {expl["narrative"]}')
    print(f'Confidence: {expl["confidence"]}')
    if expl['signature']:
        print(f'Signature: {expl["signature"]["label"]}')
        print(f'Detail: {expl["signature"]["detail"]}')
    print(f'\nRanked Features:')
    print(f'{"Rank":>4} {"Feature":25} {"Importance":>10} {"SHAP":>8} {"LSTM":>8} {"Dir":>8}')
    for i, f in enumerate(expl['ranked_features']):
        print(f'{i+1:4d} {f["feature"]:25} {f["importance"]:10.4f} '
              f'{f["shap_score"]:8.4f} {f["lstm_score"]:8.4f} {f["direction"]:>8}')

## 6. Old vs New explanation comparison

The old system used z-score (>2σ from mean → "affected"). The new system uses SHAP + LSTM fusion.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Old method: z-score
X_train = np.array([[r[f] for f in if_model.feature_names] for r in readings])
sc = StandardScaler().fit(X_train)

print(f'{"":30} {"OLD (z-score)":>35} {"NEW (SHAP+LSTM)":>35}')
print(f'{"":30} {"─"*35} {"─"*35}')

for name, anomaly_reading in test_cases:
    # Old: z-score
    x = np.array([[anomaly_reading.get(f, 0) for f in if_model.feature_names]])
    z = np.abs(sc.transform(x)).flatten()
    old_top = sorted(zip(if_model.feature_names, z), key=lambda t: -t[1])[:3]
    old_str = ', '.join(f'{f}(z={v:.1f})' for f, v in old_top)

    # New: SHAP+LSTM fusion
    if_r = if_model.score(anomaly_reading)
    window = readings[-19:] + [anomaly_reading]
    lstm_r = lstm_model.score_window(window) if lstm_model.is_trained else {
        'lstm_error': 0, 'is_anomaly': False, 'per_feature_error': []}
    expl = explainer.explain(if_r, lstm_r, baseline, top_k=3)
    new_str = ', '.join(f'{f["feature"]}(imp={f["importance"]:.2f})'
                        for f in expl['ranked_features'][:3])

    print(f'{name[:28]:30} {old_str:>35} {new_str:>35}')

print(f'\n{"─"*100}')
print('KEY DIFFERENCES:')
print('• Old: z-score is feature-independent — ranks by raw deviation, ignores model structure')
print('• New: SHAP is model-faithful — ranks by actual contribution to IF anomaly score')
print('• New: LSTM adds temporal context — catches features that deviate from recent sequence patterns')
print('• New: Fusion layer provides confidence, failure signatures, and actionable narratives')

## Summary

| Aspect | Old System | New System |
|---|---|---|
| Feature attribution method | z-score (>2σ from mean) | SHAP TreeExplainer (model-faithful) |
| Temporal context | None | LSTM per-feature reconstruction error |
| Multiple model fusion | None (IF only) | Weighted SHAP (0.6) + LSTM (0.4) |
| Confidence estimation | None | Model agreement (high/medium/low) |
| Failure classification | None | 6 signature patterns with probabilistic matching |
| Actionable output | "z > 2 on fan1" | "Probable cooling failure. Check fan bearings, dust filters." |
| Computational cost | Negligible | SHAP runs only on anomalies (<5% of readings) |

The new system transforms raw anomaly scores into **interpretable, actionable explanations with confidence levels** — moving from "something is wrong" to "this is probably wrong, here is why, and here is what to check."